# RuSearchRank, этап 2 — zero-shot reranking с cross-encoder

Cross-encoder — модель, которая совместно обрабатывает запрос и документ и оценивает их релевантность. Этот сценарий Colab восстанавливает проверенный `rusearchrank_phase1_results.zip`, загружает `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1` в неизменяемой ревизии `1427fd652930e4ba29e8149678df786c240d8825`, выполняет reranking 125 200 пар на GPU, рассчитывает nDCG@10 официальным NIST `trec_eval` v9.0.8 и создаёт `artifacts/rusearchrank_phase2_results.zip`. Он не повторяет BM25, не перестраивает кандидатов и не выполняет fine-tuning. Запускайте ячейки один раз по порядку.

In [ ]:
# Cell 2 — fail-fast Linux, GPU, RAM, and disk gate.
import os, platform, shutil, subprocess
from pathlib import Path

MIN_FREE_GIB = 15
MIN_RAM_GIB = 8
disk = shutil.disk_usage('/content' if Path('/content').is_dir() else '.')
mem_kib = next((int(line.split()[1]) for line in Path('/proc/meminfo').read_text().splitlines() if line.startswith('MemTotal:')), 0)
gpu = subprocess.run(['nvidia-smi'], text=True, capture_output=True, check=False)
print({'platform': platform.platform(), 'disk_free_gib': disk.free / 1024**3, 'ram_gib': mem_kib / 1024**2})
print(gpu.stdout or gpu.stderr)
if platform.system() != 'Linux': raise RuntimeError('Phase 2 Colab inference requires Linux.')
if gpu.returncode != 0: raise RuntimeError('A working NVIDIA GPU is required before model download.')
if disk.free < MIN_FREE_GIB * 1024**3: raise RuntimeError('Insufficient free disk for Phase 1 restore and Phase 2 outputs.')
if mem_kib < MIN_RAM_GIB * 1024**2: raise RuntimeError('Insufficient RAM for the Arrow passage table.')

In [ ]:
# Cell 3 — streaming command helper and exact branch clone/fast-forward.
import json, os, shlex, subprocess, sys, threading
from pathlib import Path

LOG_DIR = Path('/content/rusearchrank-phase2-logs')
def run_checked(command, *, cwd=None, env=None, stream=False, log_path=None, stage=None):
    command = [str(value) for value in command]
    effective_cwd = Path(cwd or Path.cwd()).resolve()
    effective_env = os.environ.copy() if env is None else env.copy()
    shown_env = {key: effective_env.get(key) for key in ('JAVA_HOME', 'PATH', 'PYTHONPATH', 'HF_HOME', 'HF_HUB_CACHE') if effective_env.get(key)}
    print('+', shlex.join(command), '\n  cwd:', effective_cwd, '\n  env:', json.dumps(shown_env), flush=True)
    if stream:
        process = subprocess.Popen(command, cwd=effective_cwd, env=effective_env, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, bufsize=1)
        stdout_lines, stderr_lines = [], []
        def pump(pipe, destination, collected):
            for line in iter(pipe.readline, ''):
                collected.append(line); print(line, end='', file=destination, flush=True)
            pipe.close()
        workers = [threading.Thread(target=pump, args=(process.stdout, sys.stdout, stdout_lines), daemon=True), threading.Thread(target=pump, args=(process.stderr, sys.stderr, stderr_lines), daemon=True)]
        [worker.start() for worker in workers]
        returncode = process.wait(); [worker.join() for worker in workers]
        stdout, stderr = ''.join(stdout_lines), ''.join(stderr_lines)
    else:
        result = subprocess.run(command, cwd=effective_cwd, env=effective_env, text=True, capture_output=True, check=False)
        returncode, stdout, stderr = result.returncode, result.stdout, result.stderr
        if stdout: print(stdout, end='' if stdout.endswith('\n') else '\n')
        if stderr: print(stderr, end='' if stderr.endswith('\n') else '\n', file=sys.stderr)
    print('  return code:', returncode, flush=True)
    record = {'stage': stage, 'command': command, 'cwd': str(effective_cwd), 'env': shown_env, 'returncode': returncode, 'stdout': stdout, 'stderr': stderr}
    log_path = Path(log_path) if log_path else LOG_DIR / (str(stage or 'command') + '.json')
    log_path.parent.mkdir(parents=True, exist_ok=True); log_path.write_text(json.dumps(record, ensure_ascii=False, indent=2) + '\n')
    print('  complete log:', log_path, flush=True)
    if returncode != 0:
        tail = ''.join((stderr or stdout or '').splitlines(keepends=True)[-25:])
        raise RuntimeError(f'stage={stage} failed with return code {returncode}\ncommand: {shlex.join(command)}\ncwd: {effective_cwd}\ncomplete log: {log_path}\nlast output lines:\n{tail}')
    return record

REPO_URL = 'https://github.com/kopanevk/ru-search-rank.git'
BRANCH = 'phase-2'
REPO_DIR = Path('/content/ru-search-rank')
ALLOW_OVERWRITE_PHASE2 = False
if (REPO_DIR / '.git').is_dir():
    run_checked(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, stage='git_fetch')
    run_checked(['git', 'checkout', BRANCH], cwd=REPO_DIR, stage='git_checkout')
    run_checked(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR, stage='git_pull')
else:
    run_checked(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], cwd='/content', stage='git_clone')
os.chdir(REPO_DIR); run_checked(['git', 'log', '-1', '--oneline', '--decorate'], cwd=REPO_DIR, stage='git_head')

In [ ]:
# Ячейка 4 — сборка зафиксированного NIST trec_eval v9.0.8.
import hashlib, json, os, re, shutil
from datetime import datetime, timezone
from pathlib import Path
run_checked(['apt-get', 'update', '-qq'], stage='apt_update')
run_checked(['apt-get', 'install', '-y', '-qq', 'build-essential'], stage='apt_build_tools')
TREC_EVAL_TAG = 'v9.0.8'
TREC_EVAL_COMMIT = 'd95ca64e14a47d763ae349fb65e6d8cde4141dbd'
TREC_EVAL_DIR = Path('/content/trec_eval-v9.0.8')
TREC_EVAL_BIN = Path('/content/rusearchrank-bin/trec_eval')
if TREC_EVAL_DIR.exists(): shutil.rmtree(TREC_EVAL_DIR)
run_checked(['git', 'clone', '--depth', '1', '--branch', TREC_EVAL_TAG, 'https://github.com/usnistgov/trec_eval.git', str(TREC_EVAL_DIR)], cwd='/content', stage='trec_eval_fresh_clone')
head = run_checked(['git', 'rev-parse', 'HEAD'], cwd=TREC_EVAL_DIR, stage='trec_head')['stdout'].strip()
tagged = run_checked(['git', 'rev-list', '-n', '1', TREC_EVAL_TAG], cwd=TREC_EVAL_DIR, stage='trec_tag')['stdout'].strip()
if head != tagged or head != TREC_EVAL_COMMIT: raise RuntimeError(f'trec_eval checkout is not exactly {TREC_EVAL_TAG}@{TREC_EVAL_COMMIT}: {head}')
run_checked(['git', 'diff', '--quiet', 'HEAD'], cwd=TREC_EVAL_DIR, stage='trec_tracked_clean_before')
run_checked(['git', 'diff', '--cached', '--quiet'], cwd=TREC_EVAL_DIR, stage='trec_index_clean_before')
makefile_sha256_before = hashlib.sha256((TREC_EVAL_DIR / 'Makefile').read_bytes()).hexdigest()
jobs = os.cpu_count() or 2; build_command = f'make -j{jobs}'
run_checked(['make', f'-j{jobs}'], cwd=TREC_EVAL_DIR, stage='trec_make')
run_checked(['git', 'diff', '--quiet', 'HEAD'], cwd=TREC_EVAL_DIR, stage='trec_tracked_clean_after')
run_checked(['git', 'diff', '--cached', '--quiet'], cwd=TREC_EVAL_DIR, stage='trec_index_clean_after')
makefile_sha256_after = hashlib.sha256((TREC_EVAL_DIR / 'Makefile').read_bytes()).hexdigest()
if makefile_sha256_after != makefile_sha256_before: raise RuntimeError('upstream Makefile changed during build')
TREC_EVAL_BIN.parent.mkdir(parents=True, exist_ok=True)
run_checked(['install', '-m', '0755', str(TREC_EVAL_DIR / 'trec_eval'), str(TREC_EVAL_BIN)], stage='trec_install')
trec_version = run_checked([str(TREC_EVAL_BIN), '-v'], stage='trec_eval_version')
if not re.search(r'\b9\.0\.7\b', trec_version['stdout'] or trec_version['stderr']): raise RuntimeError('official v9.0.8 binary must honestly report 9.0.7')
os.environ['TREC_EVAL_PATH'] = str(TREC_EVAL_BIN.resolve())
if not TREC_EVAL_BIN.is_file() or not os.access(TREC_EVAL_BIN, os.X_OK): raise RuntimeError('trec_eval не является исполняемым обычным файлом')
binary_sha256 = hashlib.sha256(TREC_EVAL_BIN.read_bytes()).hexdigest()
compiler = run_checked(['cc', '--version'], stage='trec_compiler')['stdout'].splitlines()[0]
provenance = {'source_repository': 'https://github.com/usnistgov/trec_eval.git', 'source_tag': TREC_EVAL_TAG, 'source_commit': head, 'source_tree_clean': True, 'fresh_checkout': True, 'source_path': str(TREC_EVAL_DIR), 'makefile_sha256': makefile_sha256_after, 'binary_path': str(TREC_EVAL_BIN.resolve()), 'binary_sha256': binary_sha256, 'binary_reported_version': '9.0.7', 'expected_release_version': '9.0.8', 'known_upstream_version_string_mismatch': True, 'build_command': build_command, 'compiler': compiler, 'built_at': datetime.now(timezone.utc).isoformat()}
provenance_path = REPO_DIR / 'artifacts/work/phase2/trec_eval_build_provenance.json'; provenance_path.parent.mkdir(parents=True, exist_ok=True); provenance_path.write_text(json.dumps(provenance, indent=2) + '\n', encoding='utf-8')
print(json.dumps(provenance, indent=2))

In [ ]:
# Cell 5 — isolated Python 3.12 environment and project installation.
import sys
from pathlib import Path
VENV_DIR = Path('/content/rusearchrank-phase2-py312')
if sys.version_info[:2] == (3, 12):
    run_checked(['apt-get', 'install', '-y', '-qq', 'python3.12-venv'], stage='apt_venv'); python312 = Path(sys.executable)
    if not (VENV_DIR / 'bin/python').is_file(): run_checked([str(python312), '-m', 'venv', str(VENV_DIR)], stage='create_venv')
else:
    UV_VERSION = '0.8.13'; uv_prefix = Path('/content/uv-bootstrap'); uv = uv_prefix / 'bin/uv'
    if not uv.is_file(): run_checked([sys.executable, '-m', 'pip', 'install', '--prefix', str(uv_prefix), f'uv=={UV_VERSION}'], stage='install_uv')
    run_checked([str(uv), 'python', 'install', '3.12'], stage='uv_python')
    if not (VENV_DIR / 'bin/python').is_file(): run_checked([str(uv), 'venv', '--python', '3.12', str(VENV_DIR)], stage='uv_venv')
RUN_PYTHON = VENV_DIR / 'bin/python'
run_checked([str(RUN_PYTHON), '--version'], stage='python_version')
run_checked([str(RUN_PYTHON), '-m', 'pip', 'install', '--upgrade', 'pip'], cwd=REPO_DIR, stage='pip_upgrade')
run_checked([str(RUN_PYTHON), '-m', 'pip', 'install', '-e', str(REPO_DIR)], cwd=REPO_DIR, stage='pip_install_project')

In [ ]:
# Cell 6 — complete tests, both notebook validators, and exact dependency versions.
run_checked([str(RUN_PYTHON), '-m', 'pytest', '-q'], cwd=REPO_DIR, env=os.environ, stage='pytest')
run_checked([str(RUN_PYTHON), 'scripts/validate_phase1_notebook.py'], cwd=REPO_DIR, env=os.environ, stage='validate_phase1_notebook')
run_checked([str(RUN_PYTHON), 'scripts/validate_phase2_notebook.py'], cwd=REPO_DIR, env=os.environ, stage='validate_phase2_notebook')
versions = "import importlib.metadata as m; print({name: m.version(name) for name in ('torch','transformers','tokenizers','huggingface-hub','pyarrow','pandas')})"
run_checked([str(RUN_PYTHON), '-c', versions], cwd=REPO_DIR, env=os.environ, stage='version_probe')

In [ ]:
# Cell 7 — restore and validate immutable Phase 1 inputs, then obtain dev qrels.
import hashlib, json, os, zipfile
from pathlib import Path
PHASE1_ZIP = Path(os.environ.get('PHASE1_ZIP', '/content/rusearchrank_phase1_results.zip'))
if not PHASE1_ZIP.is_file(): raise RuntimeError('Set PHASE1_ZIP to the Colab/Drive path of rusearchrank_phase1_results.zip.')
with zipfile.ZipFile(PHASE1_ZIP) as archive:
    if archive.testzip() is not None: raise RuntimeError('Phase 1 ZIP failed CRC validation.')
    for info in archive.infolist():
        member = Path(info.filename)
        if member.is_absolute() or '..' in member.parts: raise RuntimeError(f'Unsafe Phase 1 ZIP member: {info.filename}')
    archive.extractall(REPO_DIR)
manifest_path = REPO_DIR / 'reports/audit/candidate_cache_manifest.json'
manifest = json.loads(manifest_path.read_text())
if manifest.get('status') != 'PASS': raise RuntimeError('Restored Phase 1 manifest is not PASS.')
for entry in manifest['artifacts']:
    path = REPO_DIR / entry['path']; digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if path.stat().st_size != entry['size_bytes'] or digest != entry['sha256']: raise RuntimeError(f'Phase 1 hash mismatch: {entry["path"]}')
CONFIG = 'configs/rerank.yaml'
def cli(*arguments, stream=False):
    stage = '__'.join(str(value) for value in arguments).replace('/', '_')
    return run_checked([str(RUN_PYTHON), '-m', 'rusearchrank.cli', *arguments], cwd=REPO_DIR, env=os.environ, stream=stream, log_path=REPO_DIR / 'artifacts/work/phase2/notebook_logs' / (stage + '.json'), stage=stage)
cli('prepare-annotations', '--config', 'configs/retrieval.yaml')

In [ ]:
# Cell 8 — rerank preflight: config, Phase 1 hashes, model SHA, qrels, trec_eval, disk, RAM.
cli('preflight', '--stage', 'rerank', '--config', CONFIG)

In [ ]:
# Cell 9 — real pinned-model smoke on 64 real pairs; mandatory before scoring.
smoke = cli('smoke-rerank', '--config', CONFIG, '--limit', '64', '--device', 'auto', stream=True)
lines = smoke['stdout'].splitlines(); start = max(index for index, line in enumerate(lines) if line.rstrip() == '{')
smoke_report = json.loads('\n'.join(lines[start:]))
RERANK_SMOKE_PASSED = smoke_report.get('status') == 'PASS' and smoke_report.get('real_model_forward') is True and smoke_report.get('fixture_only') is False
if not RERANK_SMOKE_PASSED: raise RuntimeError('Real rerank smoke failed; do not run Cell 10.')

In [ ]:
# Cell 10 — heavy GPU scoring of all 125,200 dev pairs with shard resume.
if not globals().get('RERANK_SMOKE_PASSED'): raise RuntimeError('Cell 9 real smoke must pass first.')
overwrite = ['--overwrite'] if ALLOW_OVERWRITE_PHASE2 else []
cli('rerank-score', '--config', CONFIG, '--split', 'dev', '--device', 'auto', *overwrite, stream=True)

In [ ]:
# Cell 11 — official K=100 and diagnostic K=10/20/50 rank-preserving TREC runs.
overwrite = ['--overwrite'] if ALLOW_OVERWRITE_PHASE2 else []
for depth in (100, 10, 20, 50): cli('build-rerank-run', '--config', CONFIG, '--split', 'dev', '--depth', str(depth), *overwrite)

In [ ]:
# Cell 12 — NIST evaluation, BM25 recomputation, bootstrap, sparse diagnostics, depth profile.
overwrite = ['--overwrite'] if ALLOW_OVERWRITE_PHASE2 else []
cli('evaluate-rerank', '--config', CONFIG, '--split', 'dev', *overwrite, stream=True)

In [ ]:
# Cell 13 — byte-exact protocol snapshot, non-self-referential manifest, validated ZIP.
overwrite = ['--overwrite'] if ALLOW_OVERWRITE_PHASE2 else []
cli('package-phase2', '--config', CONFIG, *overwrite, stream=True)

In [ ]:
# Cell 14 — stream SHA-256, exact ZIP members, optional Drive copy, and download.
import hashlib, shutil, zipfile
archive_path = REPO_DIR / 'artifacts/rusearchrank_phase2_results.zip'
digest = hashlib.sha256()
with archive_path.open('rb') as stream:
    for chunk in iter(lambda: stream.read(1024 * 1024), b''): digest.update(chunk)
with zipfile.ZipFile(archive_path) as archive: contents = archive.namelist()
print({'size_bytes': archive_path.stat().st_size, 'sha256': digest.hexdigest(), 'contents': contents})
DRIVE_DESTINATION = os.environ.get('PHASE2_DRIVE_DESTINATION', '')
if DRIVE_DESTINATION: shutil.copy2(archive_path, DRIVE_DESTINATION); print('Copied to', DRIVE_DESTINATION)
try:
    from google.colab import files
except ImportError: print('Retrieve the validated ZIP from', archive_path)
else: files.download(str(archive_path))